# Single-Modality (Tabular Only) Models

In [12]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, cohen_kappa_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [3]:
processed_data = pd.read_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-alladults.csv')
processed_data.head()

,race,gender,age_at_visit,temperature,heartrate,resprate,sbp,dbp,o2sat,pain,unable,chiefcomplaint,acuity
0,Black/African American Only,0,30.0,98.3,128.0,16.0,149.0,79.0,100.0,10.0,0,"Carbuncle, furuncle, boil, cellulitis...",3
1,Black/African American Only,0,64.0,98.0,106.0,18.0,142.0,111.0,100.0,4.0,0,Painful urination,3
2,Blank,1,73.0,101.4,93.0,NaN,112.0,49.0,97.0,0.0,0,Other symptoms referable to the respi...,4
3,Black/African American Only,0,69.0,98.4,107.0,18.0,101.0,63.0,100.0,NaN,1,Other symptoms referable to the respi...,5
4,Blank,0,51.0,97.9,74.0,18.0,114.0,84.0,97.0,7.0,0,"Abdominal pain, cramps, spasms, NOS",3


In [23]:
def xgboost_standard(data):
    X = data.drop(columns=['chiefcomplaint', 'race', 'acuity'])
    Y = data['acuity']
    
    Y_shifted = Y - 1  # shift down by 1 for XGBoost
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y_shifted, test_size=0.2, random_state=0, stratify=Y_shifted
    )
    
    classifier = xgb.XGBClassifier(
        n_estimators=500, 
        learning_rate=0.05,
        objective='multi:softmax',
        num_class=5, # (0 through 4)
        eval_metric='mlogloss',
        early_stopping_rounds=25,
        random_state=0
    )
    
    classifier.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose=False)

    results = classifier.evals_result()
    final_train_error = results['validation_0']['mlogloss'][-1]
    final_test_error = results['validation_1']['mlogloss'][-1]
    
    print("XGBoost Classification Results:")
    print(f"Final Training Error (mlogloss): {final_train_error:.4f}")
    print(f"Final Testing Error (mlogloss): {final_test_error:.4f}\n")

    predicted = classifier.predict(X_test)
    
    print(classification_report(y_test, predicted))
    
    # primary metric
    kappa = cohen_kappa_score(y_test, predicted, weights='quadratic')
    print(f"Cohen Kappa Score (Quadratic): {kappa:.4f}\n")
    
    return classifier

In [24]:
def xgboost_regression(data):
    X = data.drop(columns=['chiefcomplaint', 'race', 'acuity'])
    Y = data['acuity']
    
    # stratify
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.2, random_state=0, stratify=Y)
    
    regressor = xgb.XGBRegressor(
        n_estimators=500, 
        learning_rate=0.05,
        objective='reg:squarederror',
        early_stopping_rounds=25,
        random_state=0
    )
    
    regressor.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose=False)

    results = regressor.evals_result()
    final_train_error = results['validation_0']['rmse'][-1]
    final_test_error = results['validation_1']['rmse'][-1]
    
    print("XGBoost Regression Results:")
    print(f"Final Training Error (RMSE): {final_train_error:.4f}")
    print(f"Final Testing Error (RMSE): {final_test_error:.4f}\n")

    predicted = regressor.predict(X_test)
    
    # round predictions to nearest integer 
    rounded = np.round(predicted).astype(int)
    rounded = np.clip(rounded, 1, 5) 
    y_test_int = y_test.astype(int)
    
    print(classification_report(y_test_int, rounded, labels=[1, 2, 3, 4, 5]))
    
    kappa = cohen_kappa_score(y_test_int, rounded, weights='quadratic')
    print(f"Cohen Kappa Score (Quadratic): {kappa:.4f}\n")
    
    return regressor

In [25]:

print("Multiclass Classification Model:")
clf_model = xgboost_standard(processed_data)

Multiclass Classification Model:
XGBoost Classification Results:
Final Training Error (mlogloss): 0.8880
Final Testing Error (mlogloss): 0.9124

              precision    recall  f1-score   support

           0       0.71      0.60      0.65      4809
           1       0.56      0.29      0.39     28654
           2       0.60      0.87      0.71     47935
           3       0.48      0.05      0.09      7324
           4       0.50      0.00      0.01       445

    accuracy                           0.60     89167
   macro avg       0.57      0.36      0.37     89167
weighted avg       0.58      0.60      0.55     89167

Cohen Kappa Score (Quadratic): 0.4244



In [26]:
print("Regression Model:")
reg_model = xgboost_regression(processed_data)

Regression Model:
XGBoost Regression Results:
Final Training Error (RMSE): 0.6121
Final Testing Error (RMSE): 0.6207

              precision    recall  f1-score   support

           1       0.76      0.51      0.61      4809
           2       0.54      0.29      0.38     28654
           3       0.60      0.88      0.71     47935
           4       0.50      0.04      0.07      7324
           5       0.00      0.00      0.00       445

    accuracy                           0.60     89167
   macro avg       0.48      0.34      0.35     89167
weighted avg       0.58      0.60      0.54     89167

Cohen Kappa Score (Quadratic): 0.4129



/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [13]:
heldout_data = pd.read_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-pedsNHAMCS.csv')

X_heldout = heldout_data.drop(columns=['chiefcomplaint', 'race', 'acuity'])
Y_heldout = heldout_data['acuity']

=== Evaluating on Held-Out Pediatric Dataset ===


In [19]:
print("Multiclass Classification Model (peds):")

Y_heldout_shifted = Y_heldout - 1 

clf_predictions = clf_model.predict(X_heldout)

print(classification_report(Y_heldout_shifted, clf_predictions))
clf_kappa = cohen_kappa_score(Y_heldout_shifted, clf_predictions, weights='quadratic')
print(f"Held-out Cohen Kappa (Classification): {clf_kappa:.4f}")

Multiclass Classification Model (peds):
              precision    recall  f1-score   support

           0       1.00      0.16      0.28        49
           1       0.79      0.15      0.25       572
           2       0.54      0.53      0.54      2142
           3       0.56      0.76      0.64      2432
           4       0.94      0.12      0.22       386

    accuracy                           0.56      5581
   macro avg       0.76      0.34      0.38      5581
weighted avg       0.60      0.56      0.53      5581

Held-out Cohen Kappa (Classification): 0.3018


In [ ]:
print("Regression Model (peds):")
reg_predictions = reg_model.predict(X_heldout)

# round again
reg_rounded = np.round(reg_predictions).astype(int)
reg_rounded = np.clip(reg_rounded, 1, 5)

print(classification_report(Y_heldout, reg_rounded))
reg_kappa = cohen_kappa_score(Y_heldout, reg_rounded, weights='quadratic')
print(f"Cohen Kappa (Regression): {reg_kappa:.4f}")

Regression Model (peds):
              precision    recall  f1-score   support

           1       1.00      0.02      0.04        49
           2       0.72      0.05      0.09       572
           3       0.47      0.66      0.55      2142
           4       0.55      0.58      0.56      2432
           5       0.00      0.00      0.00       386

    accuracy                           0.51      5581
   macro avg       0.55      0.26      0.25      5581
weighted avg       0.50      0.51      0.47      5581

Held-out Cohen Kappa (Regression): 0.2968


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}